# MIDA507 -- Session 07
## Ethique, Anonymisation et Protection des Donnees

| | |
|---|---|
| **Programme** | Master MIDA -- Universite de Douala |
| **Session** | S07 -- Ethique, k-anonymite et protection des donnees |
| **Duree estimee** | 90 minutes |
| **Prerequis** | Notebooks S01 a S06 |

### Ce que vous allez apprendre
1. Identifier les donnees personnelles dans un jeu documentaire
2. Connaitre le cadre juridique (RGPD + loi camerounaise 2010/012)
3. Comprendre la difference fondamentale entre pseudonymisation et anonymisation
4. Appliquer la k-anonymite (k=5) sur notre jeu de bibliotheque
5. Produire un rapport ethique et le jeu anonymise publiable

### Livrables : `MIDA507_S07_jeu_anonymise.csv` + `MIDA507_S07_rapport_ethique.json`


In [ ]:
!pip install pandas seaborn matplotlib openpyxl --quiet
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
import json, os, random, hashlib
from datetime import datetime, timedelta, date
plt.rcParams['figure.figsize']=(12,5)
sns.set_theme(style='whitegrid')
print('Outils prets.')


In [ ]:
random.seed(42)
NB=5000
TYPES=["These et memoire","Manuel universitaire","Revue scientifique",
       "Rapport de recherche","Ouvrage de reference","Document administratif"]
FIL=["Droit","Sciences eco.","Lettres","Histoire","Geographie","Medecine","Agronomie","Informatique"]
NIV=["Licence 1","Licence 2","Licence 3","Master 1","Master 2","Doctorat"]
REG=["Adamaoua","Centre","Est","Extreme-Nord","Littoral","Nord","Ouest","Sud"]
LNG=["Francais","Anglais","Bilingue","Arabe","Autres"]
d0=datetime(2018,1,1)
emprunts=pd.DataFrame({
    "cote":[f"UNG-{str(i+1).zfill(5)}" for i in range(NB)],
    "type_doc":random.choices(TYPES,weights=[0.28,0.30,0.15,0.10,0.10,0.07],k=NB),
    "date":[(d0+timedelta(days=random.randint(0,365*5))).strftime("%Y-%m-%d") for _ in range(NB)],
    "duree":random.choices([3,7,7,14,14,14,21,30],k=NB),
    "usager":[f"USR{str(random.randint(1,800)).zfill(4)}" for _ in range(NB)],
    "filiere":random.choices(FIL,k=NB),
    "niveau":random.choices(NIV,k=NB),
    "region":random.choices(REG,k=NB),
    "langue":random.choices(LNG,weights=[0.62,0.22,0.10,0.04,0.02],k=NB),
})
emprunts["annee"]=pd.to_datetime(emprunts["date"]).dt.year
print(f"Jeu BU-UNG : {len(emprunts):,} emprunts, {len(emprunts.columns)} colonnes.")


---
## NOTION 1 -- Qu'est-ce qu'une Donnee Personnelle ?

### Definition en 3 lignes

Une **donnee personnelle** est toute information qui permet d'identifier directement ou indirectement une personne physique. Directement : nom, prenom, numero de telephone. Indirectement : une combinaison de caracteristiques qui permet de singulariser une personne.

**Analogie IDA :** Dans un catalogue de lecteurs, le nom du lecteur EST une donnee personnelle. Son code de lecteur EST une donnee personnelle. Sa filiere et son niveau combiness avec sa region d'origine PEUVENT constituer une donnee personnelle si cette combinaison est rare dans le fonds.

**La question cle que doit se poser l'IDA AVANT de publier :**
> "Est-ce qu'en consultant ce jeu, quelqu'un pourrait identifier une personne physique precise ?"

Si la reponse est OUI ou PEUT-ETRE, il faut anonymiser avant de publier.


In [ ]:
# NOTION 1 EN PRATIQUE -- Identifier les donnees personnelles dans notre jeu
print("AUDIT DES DONNEES PERSONNELLES -- Jeu BU-UNG")
print("=" * 60)
print()

colonnes_analysees = {
    "cote":     {"type":"Identifiant du document (pas de la personne)","risque":"NUL","action":"Garder tel quel"},
    "type_doc": {"type":"Categorie documentaire","risque":"NUL","action":"Garder tel quel"},
    "date":     {"type":"Date d'emprunt (evenement, pas attribut de personne)","risque":"FAIBLE","action":"Generaliser en semestre si risque de linkage"},
    "duree":    {"type":"Duree (statistique d'usage)","risque":"FAIBLE","action":"Garder -- utile pour les analyses"},
    "usager":   {"type":"Code usager (identifiant DIRECT d'une personne)","risque":"ELEVE","action":"Supprimer ou pseudonymiser fortement"},
    "filiere":  {"type":"Quasi-identifiant (attribut de la personne)","risque":"MOYEN","action":"Generaliser en domaines disciplinaires"},
    "niveau":   {"type":"Quasi-identifiant (attribut de la personne)","risque":"MOYEN","action":"Generaliser en cycles LMD"},
    "region":   {"type":"Quasi-identifiant (attribut de la personne)","risque":"MOYEN","action":"Potentiellement garder si k >= 5"},
    "langue":   {"type":"Attribut culturel (potentiellement sensible)","risque":"FAIBLE","action":"Garder -- trop agregee pour identifier"},
    "annee":    {"type":"Donnee temporelle agregee","risque":"NUL","action":"Garder tel quel"},
}

for col, info in colonnes_analysees.items():
    icone = "DANGER" if info["risque"]=="ELEVE" else ("ATTENTION" if info["risque"] in ["MOYEN","FAIBLE"] else "OK")
    print(f"  [{icone:9}] {col:<15} : {info['type']}")
    print(f"              Risque  : {info['risque']}")
    print(f"              Action  : {info['action']}")
    print()

print("CONCLUSION AVANT PUBLICATION :")
print("  3 colonnes necessitent une action specifique :")
print("  1. 'usager'  (code lecteur) --> SUPPRIMER avant toute publication")
print("  2. 'filiere' (quasi-id)     --> GENERALISER en domaines disciplinaires")
print("  3. 'niveau'  (quasi-id)     --> GENERALISER en cycles LMD")


---
## NOTION 2 -- Cadre Juridique : RGPD et Loi Camerounaise

### Definition en 3 lignes

La **protection des donnees personnelles** est encadree par des lois contraignantes. Publier des donnees personnelles sans respecter ces lois expose l'institution a des sanctions penales et civiles. L'IDA doit connaitre les textes applicables AVANT de publier.

**Analogie IDA :** C'est l'equivalent du **secret professionnel** du bibliothecaire (les lecteurs ont le droit au respect de leur vie privee, meme en bibliotheque). Ce principe existe depuis longtemps -- il s'applique maintenant aux donnees numeriques.

### Deux textes s'appliquent simultanement a la BU-UNG :

| Texte | Champ d'application | Autorite de controle | Sanction max |
|---|---|---|---|
| **Loi camerounaise 2010/012** | Tout traitement au Cameroun | ART (Agence de Regulation des Telecommunications) | Emprisonnement 1-5 ans |
| **RGPD (UE 2016/679)** | Si des ressortissants europeens sont concernes | CNIL (France) | 4% du CA ou 20M EUR |

**Pour la BU-UNG, le RGPD s'applique-t-il ?**
Oui, si des etudiants en echange europeen (Erasmus, etc.) empruntent des documents. Meme un seul etudiant europeen suffit a declencher le RGPD.


In [ ]:
# NOTION 2 EN PRATIQUE -- Les 6 principes juridiques a respecter
print("PRINCIPES JURIDIQUES DE PROTECTION DES DONNEES")
print("(RGPD Article 5 + Loi camerounaise 2010/012 Article 3)")
print("=" * 65)
print()
principes_juridiques = [
    {
        "numero": 1,
        "nom": "Licheite et loyaute",
        "definition": "Les donnees sont collectees de maniere legale et transparente -- les usagers savent que leurs emprunts sont enregistres.",
        "BU_UNG": "OK si le reglement interieur de la bibliotheque mentionne l'enregistrement des emprunts."
    },
    {
        "numero": 2,
        "nom": "Finalite determinee",
        "definition": "Les donnees sont utilisees uniquement pour l'objectif pour lequel elles ont ete collectees.",
        "BU_UNG": "Les donnees d'emprunt sont collectees pour la gestion des prets -- les publier en open data statistique est un NOUVEL usage qui necessite une base juridique."
    },
    {
        "numero": 3,
        "nom": "Minimisation",
        "definition": "On ne collecte que les donnees strictement necessaires a la finalite.",
        "BU_UNG": "Pour l'open data statistique, le code usager est inutile -- on peut le supprimer."
    },
    {
        "numero": 4,
        "nom": "Exactitude",
        "definition": "Les donnees sont exactes et mises a jour.",
        "BU_UNG": "Travail de qualite realise en S05."
    },
    {
        "numero": 5,
        "nom": "Conservation limitee",
        "definition": "Les donnees identifiantes ne sont pas conservees indefiniment.",
        "BU_UNG": "La politique de DMP (S02) doit preciser la duree maximale de conservation des codes usagers."
    },
    {
        "numero": 6,
        "nom": "Securite",
        "definition": "Les donnees sont protegees contre les acces non autorises et les alterations.",
        "BU_UNG": "Checksums SHA-256 (S02), acces restreint au fichier d'origine, HTTPS sur CKAN (S06)."
    }
]
for p in principes_juridiques:
    print(f"  Principe {p['numero']} -- {p['nom'].upper()}")
    print(f"  Definition  : {p['definition']}")
    print(f"  BU-UNG      : {p['BU_UNG']}")
    print()


---
## NOTION 3 -- Pseudonymisation vs Anonymisation : la distinction essentielle

### Definition en 3 lignes

**Pseudonymisation :** on remplace les identifiants directs par des codes. Le lien avec la personne reelle peut etre retabli si on a la table de correspondance. La donnee reste personnelle juridiquement.

**Anonymisation :** on supprime tout lien possible avec la personne, de facon irreversible. La donnee n'est plus une donnee personnelle.

**Pour l'open data : seule l'anonymisation suffit.** La pseudonymisation protege contre les accidentels, pas contre un attaquant determine qui dispose d'autres sources.

**Analogie IDA :** La pseudonymisation, c'est cacher le nom d'un lecteur derriere un numero de dossier -- mais si quelqu'un accede a la table de correspondance, le secret est leve. L'anonymisation, c'est detruire la table de correspondance -- le lien est definitiveement rompu.


In [ ]:
# NOTION 3 EN PRATIQUE -- Demontrer la difference
print("DEMONSTRATION : Pseudonymisation vs Anonymisation")
print("=" * 60)
print()
# Exemple : Donnee originale
usager_original = "Marie Nguemo, Etudiante Master 2 Droit, Yaoundé"
print(f"DONNEE ORIGINALE : {usager_original}")
print()

# PSEUDONYMISATION : hachage MD5 du code usager
# On remplace le code par son empreinte -- reversible si on a la table
code_usager = "USR0042"
empreinte_md5 = hashlib.md5(code_usager.encode()).hexdigest()
print(f"PSEUDONYMISATION :")
print(f"  Code usager original  : {code_usager}")
print(f"  Code usager hache MD5 : {empreinte_md5}")
print(f"  Reversible ?          : OUI -- si on a la liste des codes, on peut recalculer")
print(f"                         toutes les empreintes et retrouver qui est qui.")
print(f"  Statut juridique      : Reste une DONNEE PERSONNELLE (RGPD + loi 2010/012)")
print()

# ANONYMISATION : suppression + generalisation
print("ANONYMISATION COMPLETE :")
print("  1. Supprimer la colonne 'usager' entierement --> plus d'identifiant direct")
print("  2. Generaliser 'filiere' --> 'domaine disciplinaire' (ex: Droit --> Sciences juridiques)")
print("  3. Generaliser 'niveau'  --> 'cycle LMD' (ex: Master 2 --> Master)")
print("  4. Generaliser 'date'    --> 'semestre' (ex: 2022-03-15 --> 2022-S1)")
print()
print("  Reversible ?          : NON -- impossible de retrouver l'individu")
print("  Statut juridique      : N'est PLUS une donnee personnelle --> publiable")
print()
print("CONCLUSION PRATIQUE :")
print("  Notre code 'USR0042' (pseudonymise en S01) n'est PAS suffisant.")
print("  Il faut supprimer entierement la colonne 'usager' avant publication.")
print("  C'est ce que nous allons faire maintenant.")


---
## NOTION 4 -- La k-Anonymite : garantir l'anonymat dans un tableau

### Definition en 3 lignes

La **k-anonymite** garantit que chaque individu dans un jeu de donnees est indistinguable d'au moins k-1 autres individus ayant les memes quasi-identifiants. Si k=5, il est impossible de distinguer un individu parmi au moins 5 personnes.

**Analogie IDA :** C'est le principe du **caviardage dans les archives** -- on masque les informations qui permettraient d'identifier une personne specifique, mais on conserve suffisamment d'informations pour que le document reste utile a la recherche.

**Quasi-identifiants :** les colonnes qui, seules ne permettent pas d'identifier, mais combininees le permettent (filiere + niveau + region = potentiellement identifiant si la combinaison est rare).


In [ ]:
# NOTION 4 EN PRATIQUE -- Calculer et appliquer la k-anonymite
print("CALCUL DE LA k-ANONYMITE -- Avant anonymisation")
print("=" * 60)
print()

# Les quasi-identifiants sont les colonnes qui, combinees, peuvent identifier
QUASI_IDENTIFIANTS = ["filiere","niveau","region"]

# Compter la taille de chaque groupe (filiere + niveau + region)
groupes = emprunts.groupby(QUASI_IDENTIFIANTS).size().reset_index(name="taille_groupe")
k_min_avant = groupes["taille_groupe"].min()
nb_groupes_ok = (groupes["taille_groupe"] >= 5).sum()
nb_groupes_total = len(groupes)

print(f"  Quasi-identifiants utilises : {QUASI_IDENTIFIANTS}")
print(f"  Nombre de combinaisons distinctes : {nb_groupes_total}")
print(f"  k minimum actuel : {k_min_avant}")
print(f"  Groupes avec k >= 5 : {nb_groupes_ok}/{nb_groupes_total} ({nb_groupes_ok/nb_groupes_total*100:.0f}%)")
print()
if k_min_avant < 5:
    print(f"  PROBLEME : {k_min_avant} personne(s) sont dans un groupe UNIQUE.")
    print(f"  Si quelqu'un connait la filiere, le niveau et la region de cette personne,")
    print(f"  il peut identifier EXACTEMENT qui elle est dans le jeu.")
    print(f"  --> Risque de re-identification ELEVE -- anonymisation obligatoire.")
    petits_groupes = groupes[groupes["taille_groupe"] < 5].head(3)
    print()
    print("  Exemples de groupes a risque (taille < 5) :")
    print(petits_groupes.to_string(index=False))
print()
print("On va maintenant generaliser les quasi-identifiants pour atteindre k >= 5.")


In [ ]:
# Application de l'anonymisation k=5
print("APPLICATION DE L'ANONYMISATION k=5")
print("=" * 60)
print()
emprunts_anon = emprunts.copy()
log_anon = []

# TRANSFORMATION 1 : Supprimer la colonne 'usager' (identifiant direct)
emprunts_anon = emprunts_anon.drop(columns=["usager"])
log_anon.append({"transformation":"Suppression","colonne":"usager",
                 "avant":"Code lecteur direct (USR0042)","apres":"Colonne supprimee",
                 "raison":"Identifiant direct d'une personne -- suppression obligatoire"})
print("  T1 [Suppression]    : Colonne 'usager' supprimee")

# TRANSFORMATION 2 : Generaliser 'filiere' en domaines disciplinaires
mapping_domaines = {
    "Droit":          "Sciences juridiques",
    "Sciences eco.":  "Sciences economiques et gest.",
    "Lettres":        "Lettres et arts",
    "Histoire":       "Sciences humaines et soc.",
    "Geographie":     "Sciences humaines et soc.",
    "Medecine":       "Sciences de la sante",
    "Agronomie":      "Sciences de la vie",
    "Informatique":   "Sciences exactes et tech.",
}
nb_avant_filiere = emprunts_anon["filiere"].nunique()
emprunts_anon["domaine"] = emprunts_anon["filiere"].map(mapping_domaines).fillna("Autre")
emprunts_anon = emprunts_anon.drop(columns=["filiere"])
log_anon.append({"transformation":"Generalisation","colonne":"filiere",
                 "avant":f"{nb_avant_filiere} filieres precisees",
                 "apres":f"{emprunts_anon['domaine'].nunique()} domaines disciplinaires",
                 "raison":"Quasi-identifiant -- generalisation pour atteindre k >= 5"})
print(f"  T2 [Generalisation] : 'filiere' ({nb_avant_filiere} valeurs) --> 'domaine' ({emprunts_anon['domaine'].nunique()} valeurs)")

# TRANSFORMATION 3 : Generaliser 'niveau' en cycles LMD
mapping_cycles = {
    "Licence 1":"Licence","Licence 2":"Licence","Licence 3":"Licence",
    "Master 1":"Master","Master 2":"Master","Doctorat":"Doctorat"
}
emprunts_anon["cycle"] = emprunts_anon["niveau"].map(mapping_cycles).fillna("Autre")
emprunts_anon = emprunts_anon.drop(columns=["niveau"])
log_anon.append({"transformation":"Generalisation","colonne":"niveau",
                 "avant":"6 niveaux precis (Licence 1 a Doctorat)",
                 "apres":"3 cycles LMD (Licence / Master / Doctorat)",
                 "raison":"Quasi-identifiant -- generalisation pour atteindre k >= 5"})
print(f"  T3 [Generalisation] : 'niveau' (6 valeurs) --> 'cycle' (3 valeurs)")

# TRANSFORMATION 4 : Generaliser 'date' en semestre
mois = pd.to_datetime(emprunts_anon["date"]).dt.month
emprunts_anon["semestre"] = emprunts_anon["annee"].astype(str) + "-S" + mois.apply(lambda m:"1" if m<=6 else "2")
emprunts_anon = emprunts_anon.drop(columns=["date","annee"])
log_anon.append({"transformation":"Generalisation","colonne":"date",
                 "avant":"Date precise AAAA-MM-JJ","apres":"Semestre (ex: 2022-S1)",
                 "raison":"Date precise = quasi-identifiant fort si croisee avec les autres colonnes"})
print(f"  T4 [Generalisation] : 'date' (date precise) --> 'semestre' (ex: 2022-S1)")

# Verification k-anonymite APRES
QUASI_APRES = [c for c in ["domaine","cycle","region"] if c in emprunts_anon.columns]
if QUASI_APRES:
    groupes_apres = emprunts_anon.groupby(QUASI_APRES).size().reset_index(name="taille")
    k_min_apres = groupes_apres["taille"].min()
    print()
    print(f"  k-anonymite AVANT : k = {k_min_avant}  (INSUFFISANT si < 5)")
    print(f"  k-anonymite APRES : k = {k_min_apres}  ({'OK -- CONFORME' if k_min_apres>=5 else 'ENCORE INSUFFISANT'})")
    if k_min_apres >= 5:
        print(f"  RESULTAT : Chaque individu est 'noye' dans un groupe d'au moins {k_min_apres} personnes.")
        print(f"  Le jeu est maintenant anonymise et publiable en open data.")

# Sauvegarde
emprunts_anon.to_csv("MIDA507_S07_jeu_anonymise.csv",index=False,encoding="utf-8-sig")
print()
print(f"  Colonnes AVANT  : {list(emprunts.columns)}")
print(f"  Colonnes APRES  : {list(emprunts_anon.columns)}")
print(f"  Lignes          : {len(emprunts_anon):,} (inchangees)")
print()
print("OK Jeu anonymise sauvegarde : MIDA507_S07_jeu_anonymise.csv")


---
## NOTION 5 -- Le Rapport Ethique : document de validation avant publication

### En 3 lignes

Le **rapport ethique** est le document formel qui certifie qu'un jeu de donnees a ete examine sous l'angle de la protection des donnees et que les mesures d'anonymisation appliquees sont suffisantes pour une publication en open data. Il est signe par le data steward IDA.

**Analogie IDA :** C'est l'equivalent de la **fiche d'elimination** en archivistique -- un document qui trace et certifie une decision professionnelle irrevocable et en engage la responsabilite de l'IDA.


In [ ]:
# NOTION 5 EN PRATIQUE -- Generer le rapport ethique complet
rapport_ethique = {
    "jeu_de_donnees": "BU-UNG Emprunts 2018-2023",
    "date_rapport": date.today().isoformat(),
    "responsable": "Data Steward IDA -- MIDA507",
    "institution": "Bibliotheque Centrale -- Universite de Ngaoundere",

    "cadre_juridique": {
        "lois_applicables": ["Loi camerounaise 2010/012 sur la cybersecurite",
                             "RGPD (UE) pour les ressortissants europeens eventuels"],
        "autorite_controle": "ART Cameroun",
        "base_juridique_publication": "Interet public -- Recherche statistique -- CC-BY 4.0"
    },

    "donnees_personnelles_avant": {
        "colonne_usager": "Code lecteur direct -- IDENTIFIANT DIRECT d'une personne",
        "quasi_identifiants": ["filiere","niveau","region -- combinaison potentiellement identifiante"],
        "k_anonymite_avant": int(k_min_avant) if 'k_min_avant' in dir() else "non calcule",
    },

    "mesures_anonymisation": log_anon,

    "donnees_personnelles_apres": {
        "colonnes_restantes": list(emprunts_anon.columns),
        "k_anonymite_apres": int(k_min_apres) if 'k_min_apres' in dir() else "non calcule",
        "evaluation": "Aucune donnee personnelle identifiable dans le jeu anonymise"
    },

    "avis_publication": "AUTORISEE" if ('k_min_apres' in dir() and k_min_apres>=5) else "EN ATTENTE",
    "conditions_publication": [
        "Licence CC-BY 4.0 -- attribution a la BU-UNG obligatoire",
        "Aucune tentative de re-identification n'est autorisee par la licence",
        "Ce jeu est une agregation statistique -- pas de donnees individuelles sensibles",
        "Toute utilisation commerciale des donnees est autorisee avec citation"
    ]
}

with open("MIDA507_S07_rapport_ethique.json","w",encoding="utf-8") as f:
    json.dump(rapport_ethique,f,ensure_ascii=False,indent=2)

print("RAPPORT ETHIQUE -- BU-UNG Emprunts 2018-2023")
print("=" * 60)
print(f"  Responsable : {rapport_ethique['responsable']}")
print(f"  Date        : {rapport_ethique['date_rapport']}")
print()
print("  Lois applicables :")
for loi in rapport_ethique["cadre_juridique"]["lois_applicables"]:
    print(f"    - {loi}")
print()
print("  Mesures d'anonymisation appliquees :")
for m in rapport_ethique["mesures_anonymisation"]:
    print(f"    [{m['transformation']}] {m['colonne']} : {m['avant']} --> {m['apres']}")
print()
print(f"  k-anonymite avant : {rapport_ethique['donnees_personnelles_avant']['k_anonymite_avant']}")
print(f"  k-anonymite apres : {rapport_ethique['donnees_personnelles_apres']['k_anonymite_apres']}")
print()
print(f"  AVIS FINAL : {rapport_ethique['avis_publication']}")
print()
print("OK Rapport ethique sauvegarde : MIDA507_S07_rapport_ethique.json")


---
## EXERCICE -- Evaluez le risque ethique d'un jeu de votre institution


In [ ]:
# EXERCICE -- Evaluation ethique de mon jeu de donnees
print("EVALUATION ETHIQUE -- Mon institution")
print("=" * 55)

MON_JEU = "[A COMPLETER : Nom du jeu de donnees]"

# Question 1 : Y a-t-il des donnees personnelles ?
CONTIENT_DP = None   # Remplacez par True ou False

# Question 2 : Quelles colonnes posent probleme ?
MES_COLONNES_DP = [
    "[A COMPLETER : ex. nom_etudiant]",
    "[A COMPLETER : ex. numero_telephone]",
    "[A COMPLETER : ex. combinaison age+sexe+village]",
]

# Question 3 : Quelle technique d'anonymisation recommandez-vous ?
MA_TECHNIQUE = "[A COMPLETER : Suppression / Generalisation / Agregation / k-anonymite]"

# Question 4 : Quelle loi s'applique dans votre pays ?
MA_LOI = "[A COMPLETER : ex. Loi camerounaise 2010/012]"

print(f"  Jeu de donnees : {MON_JEU}")
print(f"  Donnees perso  : {CONTIENT_DP}")
print()
if CONTIENT_DP is None:
    print("[A COMPLETER] Repondez aux 4 questions ci-dessus.")
    print()
    print("Guide :")
    print("  CONTIENT_DP = True   si votre jeu contient noms, codes, ages...")
    print("  CONTIENT_DP = False  si votre jeu ne contient QUE des agregats")
elif CONTIENT_DP:
    print("  Colonnes identifies comme problematiques :")
    for col in MES_COLONNES_DP:
        if "[A COMPLETER]" not in col:
            print(f"    ! {col}")
    print()
    print(f"  Technique recommandee : {MA_TECHNIQUE}")
    print(f"  Loi applicable        : {MA_LOI}")
    print()
    print("  CONCLUSION : Anonymisation OBLIGATOIRE avant publication.")
    print("  Sans anonymisation, la publication expose l'institution")
    print("  a des sanctions penales et civiles.")
else:
    print("  CONCLUSION : Pas de donnees personnelles -- publication possible")
    print("  apres audit FAIR (session S02) et verification de la licence (S03).")


---
## BILAN GENERAL -- Pipeline MIDA507 Complet

Vous avez maintenant parcouru les 7 sessions du cours MIDA507. Voici ce que vous savez faire :

| Session | Competence acquise | Livrable produit |
|---|---|---|
| S01 | Analyser les 7V du Big Data sur un jeu africain | Rapport 7V JSON |
| S02 | Cycle DCC, FAIR, DMP, checksums, data lineage | DMP + Rapport FAIR |
| S03 | Open data, licences, audit portails africains | Audit portail JSON |
| S04 | Metadonnees DCAT, notice HTML pour les usagers | Fiche DCAT + Notice HTML |
| S05 | Qualite DAMA : profil + nettoyage 5 regles | Jeu nettoye + Rapport qualite |
| S06 | Publication CKAN : pipeline 4 etapes + API | Package publication JSON |
| S07 | Ethique : k-anonymite + rapport de conformite | Jeu anonymise + Rapport ethique |

### Ce que vous etes capable de faire maintenant

Un professionnel IDA ayant suivi les 7 sessions MIDA507 peut :
1. **Evaluer** n'importe quel jeu de donnees africain selon les standards internationaux (FAIR, DAMA, Sunlight)
2. **Preparer** un jeu pour la publication : nettoyage, anonymisation, metadonnees DCAT
3. **Publier** sur CKAN avec tous les elements requis (licence, metadonnees, ressources)
4. **Justifier** ses decisions devant la direction, les bailleurs de fonds et les autorites de controle

*Notebook MIDA507 S07 -- Master MIDA -- Universite de Douala*
